In [ ]:
import numpy as np
import pandas as pd
from utils.load_config import load_config
from utils.load_data import verify_data_path, load_pkl

### Data loading

In [ ]:
# Obtain the path from the dataset that we want to work with
cfg = load_config()
marisma_pkl = cfg["data"]["MARISMA_PICKLE"]

In [ ]:
# Verify that it exists
verify_data_path(marisma_pkl)

In [ ]:
# Load MARISMa pickle file
marisma = load_pkl(marisma_pkl)

### Data exploration

In [ ]:
print(type(marisma))
print(marisma.keys())

In [ ]:
data, label, meta = marisma["data"], marisma["label"], marisma["meta"]

In [ ]:
print(f"Spectra matrix shape: {data.shape}")
print(f"Labels shape: {label.shape}")
print(f"Metadata shape: {meta.shape}")

In [ ]:
print(type(data))
print(type(label))
print(type(meta))

Take a closer look at each variable to familiarize with its structure

In [ ]:
print(data[:5])

The variable `data` is a 2D NumPy array of shape (n_samples, n_features).
- Rows: Individual spectra
    - Each row corresponds to one MALDI-TOF spectrum
    - That is, one bacterial isolate (sample). For example, data[0] might represent a Pseudomonas aeruginosa isolate measured in 2020.

- Columns: m/z bins
    - Each column represents an intensity value at a specific mass-to-charge (m/z) bin.
    - During preprocessing, the continuous m/z range was discretized into fixed bins (e.g., 6 000). So column j = intensity measured around a certain m/z interval.

- Values → Normalized signal intensities
    - Each number (e.g., 0.54512112) is the normalized ion intensity at that m/z bin:
        - 0.0 → no signal (flat region)
        - 0.5 → moderate signal
        - 1.0 → highest peak in that spectrum

**Summary table**

| Level        | Meaning                          | Example                         |
| ------------ | -------------------------------- | ------------------------------- |
| Row `i`      | One spectrum / bacterial isolate | *Pseudomonas aeruginosa* (2020) |
| Column `j`   | One m/z interval (binned)        | Signal around 3200 Da           |
| `data[i, j]` | Normalized intensity (0–1)       | 0.545 → medium peak strength    |


In [ ]:
print(label[:5])

In [ ]:
print(np.unique(label))

`label` is a 1D array with one entry per spectrum in data.
- Each element represents the bacterial species corresponding to that spectrum.
- Example: label[0] = "Pseudomonas_Aeruginosa" → data[0] is a spectrum from P. aeruginosa.

The dataset contains six distinct species:
- Enterobacter cloacae complex
- Enterococcus faecium
- Escherichia coli
- Klebsiella pneumoniae
- Pseudomonas aeruginosa
- Staphylococcus aureus

label[i] aligns directly with data[i] and meta[i] → all three describe the same sample.

**Summary table**
| Level                     | Meaning                                       | Example                                        |
| ------------------------- | --------------------------------------------- | ---------------------------------------------- |
| `label[i]`                | Species name of sample *i*                    | `"Pseudomonas_Aeruginosa"`                     |
| `np.unique(label)`        | All species present in the dataset            | 6 species (listed above)                       |
| Relationship to `data[i]` | Defines the class / species for that spectrum | Used for supervised training or coloring plots |

In [ ]:
print(meta)
marisma["meta"][10]

`meta` contains metadata dictionaries describing each sample (one per spectrum).
- Each row corresponds to the same sample index as data[i] and label[i].
- In the pickle, it is stored as a DataFrame with a single column of dictionaries.

**Summary table**
| Key       | Meaning                                                | Example             |
| --------- | ------------------------------------------------------ | ------------------- |
| `year`    | Year when the sample was measured                      | `'2020'`            |
| `genus`   | Bacterial genus                                        | `'Pseudomonas'`     |
| `species` | Bacterial species (same as `label`)                    | `'Aeruginosa'`      |
| `study`   | Internal study or sample ID (unique identifier / path) | `'14a578ca/0_A5/1'` |


In [ ]:
meta_expanded = pd.DataFrame.from_records(list(meta))

In [ ]:
print(meta_expanded.shape)
print(type(meta_expanded))

In [ ]:
meta_expanded.head()

In [ ]:
for col in meta_expanded.columns:
    print(f"Number of unique labels for column '{col}': {len(np.unique(meta_expanded[col]))}")
    print(f"Unique labels for column '{col}': {np.unique(meta_expanded[col])}", "\n")